# AkidaNet on the Akida simulator vs the quantized Keras CNN, on ImageNet-C

The simplest route that fits the proposal. **No training.**

1. Load BrainChip's pretrained, already quantized AkidaNet (ImageNet, alpha 0.5, 224x224, Akida 1.0, 8-bit first layer, 4-bit weights and activations).
2. Convert it with `cnn2snn` to an Akida 1.0 model (the AKD1000 target) and save the `.fbz`.
3. Run the converted model on the **Akida software simulator (CPU)**. The model is never mapped to a device. BrainChip's AkidaNet/ImageNet tutorial calls `model_akida.evaluate` this way before any `map()`.
4. Evaluate on ImageNet-C (fog, Gaussian noise, contrast, severities 1 to 5). The quantized Keras model of the same weights is the CNN baseline. Writes `results/accuracy.csv`, `agreement.csv`, `predictions.csv`, `layers.csv`.

**Wording for the write-up:** "Akida simulator, same converted model as would run on AKD1000".

**Not measured here (the simulator cannot give these):** energy per frame, power and real throughput. They exist only on the card, so this run writes no `energy.csv`. Earlier card measurements on 10 images are quoted, clearly labelled, in section 3. The clean ImageNet validation images are not available yet, so there is no clean baseline. BrainChip publishes 61.30% clean top-1 for this model, which is cited and not measured here.

**Data:** ImageNet-C (Zenodo record 2235448: `weather.tar` = fog, `digital.tar` = contrast, `noise.tar` = gaussian_noise), cut to 2 images per class with `scripts/make_imagenet_subset.py`. Layout: `IMAGENET_C_DIR/<corruption>/<severity>/<wnid>/*.JPEG`.

In [1]:
import os, csv, time
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # MetaTF tools expect Keras 2

import numpy as np
import tensorflow as tf
import akida, cnn2snn, quantizeml, akida_models
from akida_models import akidanet_imagenet_pretrained
from akida_models.imagenet import get_preprocessed_samples
from cnn2snn import convert, set_akida_version, AkidaVersion

# cnn2snn targets Akida 2.0 by default. The AKD1000 is Akida 1.0, so load and convert inside
# `with set_akida_version(AkidaVersion.v1):` (as in the MetaTF AkidaNet/ImageNet tutorial).
for m in (tf, akida, cnn2snn, quantizeml, akida_models):
    print(m.__name__, getattr(m, "__version__", "?"))

2026-11-05 16:19:25.543799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793891965.556284   18856 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793891965.559830   18856 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793891965.570240   18856 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793891965.570257   18856 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793891965.570258   18856 computation_placer.cc:177] computation placer alr

I0000 00:00:1793891967.792058   18856 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9055 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:27:00.0, compute capability: 7.5


tensorflow 2.19.1
akida 2.19.3
cnn2snn 2.19.3
quantizeml 1.2.5
akida_models 1.14.2


In [2]:
# ---- config ----
ALPHA    = 0.5      # AkidaNet width; 0.5 at 224 is the one used in BrainChip's tutorial
IMG_SIZE = 224      # the pretrained quantized model is fixed at 224

IMAGENET_C_DIR   = "../data/imagenet_subset/imagenet-c"   # <root>/<corruption>/<severity>/<wnid>/*.JPEG
IMAGENET_VAL_DIR = None     # clean val images, same file names (<root>/<wnid>/*.JPEG); not available yet

CORRUPTIONS = ["fog", "gaussian_noise", "contrast"]
SEVERITIES  = [1, 2, 3, 4, 5]
N_PER_CLASS = 2     # images per class per condition -> 2000 images per condition
CHUNK       = 250   # images per batch
MIN_AGREEMENT = 0.7 # guard on the first condition: Keras and Akida share weights, so low agreement means a bug
RESULTS_DIR = "results"

## 1. Pretrained quantized model (BrainChip zoo)

In [3]:
# Pretrained, already quantized AkidaNet (8-bit first layer, 4-bit weights, 4-bit activations), Akida 1.0 version.
with set_akida_version(AkidaVersion.v1):
    model_keras = akidanet_imagenet_pretrained(ALPHA, quantized=True, bitwidth=4)
print("input :", model_keras.input.dtype, model_keras.input_shape)
print("output:", model_keras.output_shape)

# BrainChip's 10 copyright-free sample images (224x224x3 uint8) as a first check of the Keras model
x_demo, y_demo = get_preprocessed_samples(IMG_SIZE, 3)
pots = model_keras.predict(x_demo, batch_size=100)
acc_keras_demo = np.mean(np.argmax(pots, 1) == y_demo)
print(f"TF-Keras accuracy on the {len(x_demo)} sample images: {acc_keras_demo:.2f}")

/home/yoshi/anaconda3/envs/ai4vis/lib/python3.11/site-packages/akida_models/model_io.py:147: UserWarning: Model akidanet_imagenet_224_alpha_50_iq8_wq4_aq4.h5 has been trained with akida_models 1.1.10 which is the last version supporting 1.0 models training. Continuing execution.
  warnings.warn(f'Model {model_name_v1} has been trained with akida_models 1.1.10 which is '


input : <dtype: 'float32'> (None, 224, 224, 3)
output: (None, 1000)


I0000 00:00:1793891971.485860   18990 cuda_dnn.cc:529] Loaded cuDNN version 90300


1/1 [==============================] - ETA: 0s

1/1 [==============================] - 2s 2s/step


TF-Keras accuracy on the 10 sample images: 0.90


## 2. Convert to Akida 1.0

In [4]:
with set_akida_version(AkidaVersion.v1):
    model_akida = convert(model_keras)
print("akida IP version:", model_akida.ip_version)      # v1 = Akida 1.0, the AKD1000 target
model_akida.summary()
model_akida.save("akidanet_imagenet_a05_224_v1.fbz")
print("saved akidanet_imagenet_a05_224_v1.fbz")

os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, "layers.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["index", "name", "layer_type", "input_dims", "output_dims", "input_bits", "weights_bits", "act_bits", "activation"])
    for i, l in enumerate(model_akida.layers):
        p = l.parameters
        get = lambda k: getattr(p, k) if k in dir(p) else ""   # missing keys raise ValueError, not AttributeError
        w.writerow([i, l.name, str(p.layer_type).split(".")[-1], "x".join(map(str, l.input_dims)),
                    "x".join(map(str, l.output_dims)), l.input_bits,
                    get("weights_bits"), get("act_bits"), get("activation")])
print("wrote", os.path.join(RESULTS_DIR, "layers.csv"))

akida IP version: IpVersion.v1
                 Model Summary                  
________________________________________________
Input shape    Output shape  Sequences  Layers
[224, 224, 3]  [1, 1, 1000]  1          15    
________________________________________________

_____________________________________________________________
Layer (type)              Output shape    Kernel shape     

============== SW/conv_0-classifier (Software) ==============

conv_0 (InputConv.)       [112, 112, 16]  (3, 3, 3, 16)    
_____________________________________________________________
conv_1 (Conv.)            [112, 112, 32]  (3, 3, 16, 32)   
_____________________________________________________________
conv_2 (Conv.)            [56, 56, 64]    (3, 3, 32, 64)   
_____________________________________________________________
conv_3 (Conv.)            [56, 56, 64]    (3, 3, 64, 64)   
_____________________________________________________________
separable_4 (Sep.Conv.)   [28, 28, 128]   (3, 3, 64, 

## 3. Akida simulator check (no device)

The model is **not mapped**, so `predict`/`evaluate` run on the akida software backend on the CPU. That is what BrainChip's tutorial does before its hardware section.

**Use `predict`, not `forward`, for class scores.** `forward` returns the raw int32 potentials of the last layer. `predict` returns them "obtained from the model discrete potentials by applying a shift and a scale" (akida 2.19.3 docstring), and `evaluate` scores the same way. The scale differs per output class, so the argmax of the raw `forward` output is not the predicted class. In a diagnostic on 250 fog-1 images (first image of every 4th class), the argmax of `forward` gave 22.4% top-1 and 40% agreement with Keras, while `predict` gave 36.4% top-1 and 80% agreement with Keras (Keras: 38.8%). The earlier card run used the argmax of `forward` and got the same pattern (card 0.212 top-1, 0.344 agreement on fog-1), which explains it.

**Energy, power and speed (NOT from this run).** Earlier measurements on the real AKD1000 card, 10 sample images (`model.statistics`): CLI 5.47 fps / 168 mJ per frame; Python `forward` 10.63 fps / 89 mJ per frame; notebook demo 11.19 fps / 85.5 mJ per frame; floor power ~930 mW. BrainChip's docs show ~51 fps / ~19 mJ per frame for this model, and the gap is unexplained. These are hardware measurements on 10 images, not results of this simulator run, and they are kept out of the result tables.

In [5]:
print("Akida devices visible:", akida.devices(), "(not used: everything below runs on the software simulator)")
acc_akida_demo = model_akida.evaluate(x_demo, y_demo)
pred_akida_demo = np.asarray(model_akida.predict(x_demo)).reshape(len(x_demo), -1).argmax(1)
print(f"Akida simulator accuracy on the {len(x_demo)} sample images: evaluate {acc_akida_demo:.2f}, "
      f"predict-argmax {np.mean(pred_akida_demo == y_demo):.2f} (BrainChip tutorial: 8/10)")

Akida devices visible: [] (not used: everything below runs on the software simulator)
Akida simulator accuracy on the 10 sample images: evaluate 0.80, predict-argmax 0.80 (BrainChip tutorial: 8/10)


## 4. ImageNet-C evaluation: quantized Keras (CNN baseline) vs Akida simulator

Every condition uses the same images: the first `N_PER_CLASS` files of each class. Class index = position of the class folder in sorted order, which is the standard ImageNet order. Both models get the same uint8 batches.

In [6]:
HAVE_C   = bool(IMAGENET_C_DIR)   and os.path.isdir(IMAGENET_C_DIR)
HAVE_VAL = bool(IMAGENET_VAL_DIR) and os.path.isdir(IMAGENET_VAL_DIR)

def build_sample(ref_dir, n_per_class):
    """(class_index, wnid, file name) for the first n files of every class folder in ref_dir."""
    wnids = sorted(d for d in os.listdir(ref_dir) if os.path.isdir(os.path.join(ref_dir, d)))
    assert len(wnids) == 1000, f"expected 1000 class folders in {ref_dir}, found {len(wnids)}: class indices would be wrong"
    sample = []
    for idx, wnid in enumerate(wnids):
        files = sorted(os.listdir(os.path.join(ref_dir, wnid)))[:n_per_class]
        sample += [(idx, wnid, f) for f in files]
    return sample

def load_image(path, clean):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    if clean:       # standard ImageNet eval: shorter side to 256, center crop 224
        shape = tf.cast(tf.shape(img)[:2], tf.float32)
        new_hw = tf.cast(tf.round(shape * (256.0 / tf.reduce_min(shape))), tf.int32)
        img = tf.image.resize(img, new_hw, antialias=True)
    img = tf.image.resize_with_crop_or_pad(img, IMG_SIZE, IMG_SIZE)   # ImageNet-C images are already 224x224
    return tf.cast(tf.round(tf.clip_by_value(tf.cast(img, tf.float32), 0.0, 255.0)), tf.uint8)

def make_ds(paths, labels, clean):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, y: (load_image(p, clean), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(CHUNK).prefetch(tf.data.AUTOTUNE)

def evaluate_condition(paths, labels, clean):
    """Same uint8 batch to both models. Akida: predict() on the unmapped model = software simulator."""
    keras_pred, akida_pred = [], []
    for x, _ in make_ds(paths, labels, clean):
        x = x.numpy()
        keras_pred.append(np.asarray(model_keras.predict_on_batch(x)).reshape(len(x), -1).argmax(1))
        akida_pred.append(np.asarray(model_akida.predict(x)).reshape(len(x), -1).argmax(1))
    return np.concatenate(keras_pred), np.concatenate(akida_pred)

In [7]:
rows_acc, rows_agree, rows_pred = [], [], []

def write_csv(name, header, rows):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(os.path.join(RESULTS_DIR, name), "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)

def flush():
    write_csv("accuracy.csv", ["model", "corruption", "severity", "n_images", "top1"], rows_acc)
    write_csv("agreement.csv", ["corruption", "severity", "n_images", "agreement"], rows_agree)
    write_csv("predictions.csv", ["image_id", "corruption", "severity", "label", "keras_pred", "akida_pred"], rows_pred)

if not HAVE_C:
    print("IMAGENET_C_DIR is not set (or not a folder): skipping step 4.")
else:
    sample = build_sample(os.path.join(IMAGENET_C_DIR, CORRUPTIONS[0], str(SEVERITIES[0])), N_PER_CLASS)
    labels = np.array([s[0] for s in sample], dtype=np.int64)
    n = len(sample)
    print(f"{n} images per condition")

    conditions = ([("clean", 0)] if HAVE_VAL else []) + [(c, s) for c in CORRUPTIONS for s in SEVERITIES]
    for i_cond, (corruption, severity) in enumerate(conditions):
        clean = corruption == "clean"
        root = IMAGENET_VAL_DIR if clean else os.path.join(IMAGENET_C_DIR, corruption, str(severity))
        paths = [os.path.join(root, w, f) for _, w, f in sample]
        assert os.path.isfile(paths[0]), f"missing file: {paths[0]}"
        t0 = time.time()
        k_pred, a_pred = evaluate_condition(paths, labels, clean)

        k_top1 = float(np.mean(k_pred == labels))
        a_top1 = float(np.mean(a_pred == labels))
        agree  = float(np.mean(a_pred == k_pred))
        rows_acc.append(["akidanet_keras_quantized", corruption, severity, n, f"{k_top1:.4f}"])
        rows_acc.append(["akidanet_akida_simulator", corruption, severity, n, f"{a_top1:.4f}"])
        rows_agree.append([corruption, severity, n, f"{agree:.4f}"])
        for i, (_, w, f) in enumerate(sample):
            rows_pred.append([f"{w}/{f}", corruption, severity, int(labels[i]), int(k_pred[i]), int(a_pred[i])])
        print(f"{corruption:15s} sev {severity}: keras {k_top1:.3f} | akida sim {a_top1:.3f} | agree {agree:.3f}"
              f"  ({time.time() - t0:.0f}s)")
        flush()
        if clean or severity == 1:
            assert k_top1 > 0.2, ("Keras top-1 is far below the expected ~0.6. Check class folder order, file layout "
                                  "and preprocessing before trusting any number.")
        if i_cond == 0:
            assert agree >= MIN_AGREEMENT, (f"Keras/Akida agreement {agree:.3f} < {MIN_AGREEMENT}: same weights, so "
                                            "something in conversion or feeding is wrong. Stopping.")
    print("wrote", [os.path.join(RESULTS_DIR, f) for f in ("accuracy.csv", "agreement.csv", "predictions.csv", "layers.csv")])

2000 images per condition


2026-11-05 16:19:52.606839: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


fog             sev 1: keras 0.382 | akida sim 0.377 | agree 0.787  (13s)


2026-11-05 16:20:08.706942: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


fog             sev 2: keras 0.278 | akida sim 0.282 | agree 0.742  (16s)


fog             sev 3: keras 0.199 | akida sim 0.204 | agree 0.711  (15s)


... (10 outputs left out)

contrast        sev 2: keras 0.289 | akida sim 0.293 | agree 0.753  (12s)


contrast        sev 3: keras 0.130 | akida sim 0.142 | agree 0.704  (12s)


contrast        sev 4: keras 0.029 | akida sim 0.032 | agree 0.660  (12s)


contrast        sev 5: keras 0.008 | akida sim 0.007 | agree 0.629  (11s)
wrote ['results/accuracy.csv', 'results/agreement.csv', 'results/predictions.csv', 'results/layers.csv']
